<h3/>
<h1 align="center">Universidad de Buenos Aires (FIUBA)</h1>
<h2 align="center">Especialización en Inteligencia Artificial - CEIA - FIUBA</h2>
<h2 align="center">Procesamiento del Lenguaje Natural I</h2>
<h3 align="center">Desafío 1 - Vectorización de texto y clasificación con Naïve Bayes</h3>

<p align="left">
Alumno:
  <ul>
    <li>Nicolás Lastra</li>
  </ul>
</p>

# Planteo del problema y fuente de datos

Se usa el dataset **20 Newsgroups** (incluido en scikit-learn): unas 18.000 publicaciones de grupos de noticias repartidas en 20 temas, ya separadas en train y test.

> **Cada experimento debe estar acompañado de una explicación o interpretación de lo observado**, incluido el sustento de las decisiones tomadas (por ejemplo, cambios de hiperparámetros). La notebook tiene que poder correrse de inicio a fin sin errores.

## Consigna

1. **Vectorizar documentos.** Tomar 5 documentos al azar, medir su similaridad con el resto y analizar si los 5 más similares tienen sentido según el contenido y la etiqueta.
2. **Clasificador por prototipos (tipo zero-shot).** Asignar a cada documento de test la clase del documento de train más similar.
3. **Naïve Bayes.** Maximizar el F1-score macro en test probando parámetros del vectorizador y los modelos `MultinomialNB` y `ComplementNB`. **No cambiar `ngram_range`.**
4. **Matriz término-documento.** Transponer la matriz documento-término, elegir 5 palabras **a mano** y estudiar sus 5 más similares.

---
## 0. Librerías y datos

In [1]:
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.naive_bayes import ComplementNB, MultinomialNB
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from html import escape
from IPython.display import HTML, display

nltk.download("punkt_tab") # Modelo entrenado para detectar dónde terminan las oraciones, en varios idiomas.
nltk.download("stopwords") # Listas de stopwords para unos 30 idiomas.
nltk.download("wordnet") # WordNet, es un diccionario léxico del inglés con unas 150.000 palabras, sus formas base y relaciones de sinónimos.

semilla = 42 # Mantiene reproducibilidad

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\lastr\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\lastr\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\lastr\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


---
## 1. Similaridad entre documentos

In [2]:
# Cargamos el dataset 20 Newsgroups, ya separado en train y test por la propia librería.
# remove = ("headers", "footers", "quotes") limpia cada mensaje:
#   - headers: encabezados del mail (remitente, asunto y el grupo)
#   - footers: firmas al final del mensaje
#   - quotes: líneas citadas de otros mensajes

train = fetch_20newsgroups(subset="train", remove=("headers", "footers", "quotes"))
test = fetch_20newsgroups(subset="test", remove=("headers", "footers", "quotes"))
print(f"El set de entrenamiento contiene: {len(train.data)} documentos")
print(f"El set de testeo contiene: {len(test.data)} documentos")

El set de entrenamiento contiene: 11314 documentos
El set de testeo contiene: 7532 documentos


In [3]:
# Definicion de funciones utilizadas en el item 1.

stop_words_en = set(stopwords.words("english")) # Define un conjunto con las stop words en ingles (ya que los documentos vienen en ingles).
lemmatizer = WordNetLemmatizer() #Crea el objeto que luego se utiliza para lematizar.

# Funcion de preprocesamiento del documento antes de vectorizar
def preprocesar(texto):
    tokens = word_tokenize(texto.lower())                                   # Normalizacion (pasamos a minusculas todas las palabras) + tokenization
    tokens = [t for t in tokens if t.isalpha()]                             # Remueve puntuaciones, numeros y signos raros 
    tokens = [t for t in tokens if t not in stop_words_en and len(t) > 2]   # Remueve stop words
    return [lemmatizer.lemmatize(t) for t in tokens]                        # Lematizacion


# Muestra el texto original a la izquierda y el preprocesado a la derecha.
def mostrar_preprocesado(i, max_chars=600, max_tokens=60):
    original = train.data[i][:max_chars]
    tokens = preprocesar(train.data[i])
    clase = train.target_names[train.target[i]]

    display(HTML(f"""
    <p><b>Documento {i}</b> - clase: <code>{clase}</code> - {len(tokens)} tokens después de preprocesar</p>
    <table style="width:100%; table-layout:fixed; border-collapse:collapse;">
      <tr>
        <th style="text-align:left; width:50%;">Original</th>
        <th style="text-align:left; width:50%;">Preprocesado</th>
      </tr>
      <tr>
        <td style="vertical-align:top; white-space:pre-wrap; text-align:left;">{escape(original)}</td>
        <td style="vertical-align:top; text-align:left;">{escape(" ".join(tokens[:max_tokens]))}</td>
      </tr>
    </table>
    """))

# Funcion utilizada para devolver los indices y similaridades de los k vectores mas parecidos a matriz[indice]
def top_k_similares(matriz, indice: int, k: int = 5) -> tuple[np.ndarray, np.ndarray]:
    vector_similares = cosine_similarity(matriz[indice], matriz).ravel()
    vector_similares[indice] = -np.inf #Forzamos a que la metrica de similitud con el propio documento sea la menor para que no aparezca entre los 5 mejores.
    indices_ordenados = np.argsort(vector_similares)[::-1][:k] # Almacena los 5 mejores indices ordenados de mayor a menor.
    return indices_ordenados, vector_similares[indices_ordenados]


# Para cada documento elegido, buscamos sus 5 vecinos más similares (similaridad coseno) dentro de train
# y mostramos su clase, su similaridad y el comienzo del texto, para comparar contenido y etiqueta.
def mostrar_vecinos(i, datos, etiquetas, k=5,  max_chars=200):
    indices_ordenados, vector_similares = top_k_similares(datos, i, k=k)
    clase_original = train.target_names[etiquetas[i]]

    # Palabras con más peso TF-IDF en el documento original, resumen rápido de qué trata.
    fila = datos[i].toarray().ravel()
    top_palabras = ", ".join(idx2word[j] for j in fila.argsort()[::-1][:8] if fila[j] > 0)

    filas = ""
    for j, s in zip(indices_ordenados, vector_similares):
        clase = train.target_names[etiquetas[j]]
        marca = "✔" if clase == clase_original else "✘"
        filas += f"""
        <tr>
          <td>{j}</td><td>{s:.3f}</td><td><code>{clase}</code> {marca}</td>
          <td style="text-align:left;">{escape(train.data[j][:max_chars])}</td>
        </tr>"""

    aciertos = sum(train.target_names[etiquetas[j]] == clase_original for j in indices_ordenados)
    display(HTML(f"""
    <p><b>Documento {i}</b> - clase: <code>{clase_original}</code> - {aciertos}/{k} vecinos de la misma clase<br>
    Palabras con más peso: <i>{escape(top_palabras)}</i></p>
    <table style="width:100%; table-layout:fixed; border-collapse:collapse;">
      <tr><th style="width:8%">Índice</th><th style="width:10%">Similaridad</th>
          <th style="width:27%">Clase</th><th style="text-align:left;">Comienzo del texto</th></tr>
      {filas}
    </table>
    """))


In [4]:
# Seleccionamos 5 indices random para luego extraer los documentos del train.
rng = np.random.default_rng(semilla)
indices_documentos = rng.choice(len(train.data), size=5, replace=False)

print(f"Los indices seleccionados del documento son: {indices_documentos}")
#Mostramos el texto original traido del dataset y a su derecha el mismo texto preprocesado, sin vectorizar.
for i in indices_documentos:
    mostrar_preprocesado(i)

Los indices seleccionados del documento son: [8754 4965 7404 1009 4899]


Original,Preprocesado
"/(hudson) /If someone inflicts pain on themselves, whether they enjoy it or not, they /are hurting themselves. They may be permanently damaging their body. That is true. It is also none of your business. Some people may also reason that by reading the bible and being a Xtian you are permanently damaging your brain. By your logic, it would be OK for them to come into your home, take away your bible, and send you off to ""re-education camps"" to save your mind from ruin. Are you ready for that? /(hudson) /And why is there nothing wrong with it? Because you say so? Who gave you /t",hudson someone inflicts pain whether enjoy hurting may permanently damaging body true also none business people may also reason reading bible xtian permanently damaging brain logic would come home take away bible send camp save mind ruin ready hudson nothing wrong say gave authority say set standard morality living thinking person able make choice need show think way observed many


Original,Preprocesado
"No. Plug the printer in the printer port, and the modem in the modem port. ;)",plug printer printer port modem modem port


Original,Preprocesado
"Hello. Is it possible to know minimize program manager when starting an application and to restore it when the application is ended ? If possible, please tell me how to do it !",hello possible know minimize program manager starting application restore application ended possible please tell


Original,Preprocesado
"[followups to talk.politics.guns] rl> Russell Lawrence kr> Karl Rominger kr> I support the right of any citizen with out a criminal history to own and use firearms, regardless of race, gender, and RELIGION. rl> Thanks for admitting that you, yourself, adhere to an illogical dogma. Well, folks in t.p.guns, want to show how Russell's ""illogical dogma"" is wrong?",followup russell lawrence karl rominger support right citizen criminal history use firearm regardless race gender religion thanks admitting adhere illogical dogma well folk want show russell illogical dogma wrong


Original,Preprocesado
"... Haven't you read any of Noam Chomsky's works? A widely used information net outside the control of the 'right people' is unthinkable. Hundreds of billions of dollars will be spent to wipe it out, sorry, 'regulate and order it' once the major media and poitical powers wake up to the efect it can have. If you can't be bothered reading, get the video ""Manufacturing Consent"".",read noam chomsky work widely used information net outside control right people unthinkable hundred billion dollar spent wipe sorry regulate order major medium poitical power wake efect bothered reading get video manufacturing consent


In [5]:
# Vectorizamos con TF-IDF usando el preprocesamiento propio en lugar de la tokenización de sklearn.
# El vocabulario y el IDF se aprenden solo con train, test se transforma con ese mismo vocabulario.
vectorizer = TfidfVectorizer(tokenizer=preprocesar, token_pattern=None, lowercase=False)

X_train = vectorizer.fit_transform(train.data)
X_test = vectorizer.transform(test.data)
y_train, y_test = train.target, test.target

# Diccionario inverso: de índice de columna a palabra
idx2word = {idx: palabra for palabra, idx in vectorizer.vocabulary_.items()}

n_docs, n_terminos = X_train.shape
print(f"Matriz documento-término: {n_docs} documentos × {n_terminos} términos")
print(f"Valores distintos de cero: {X_train.nnz} ({X_train.nnz / (n_docs * n_terminos):.4%} de la matriz)")

Matriz documento-término: 11314 documentos × 55460 términos
Valores distintos de cero: 669641 (0.1067% de la matriz)


In [6]:
vectorizer_default = TfidfVectorizer()
vectorizer_default.fit(train.data)
print(f"La cantidad de terminos con el preprocesamiento de sklearn es: {len(vectorizer_default.vocabulary_)}") 
print(f"La cantidad de terminos con el preprocesamiento propio es: {len(vectorizer.vocabulary_)}")          

La cantidad de terminos con el preprocesamiento de sklearn es: 101631
La cantidad de terminos con el preprocesamiento propio es: 55460


In [7]:
for i in indices_documentos:
    mostrar_vecinos(i, X_train, y_train)

Índice,Similaridad,Clase,Comienzo del texto
5966,0.391,talk.religion.misc ✔,(hudson) (me) /(hudson) /So. Why is someone elses will such a big deal if morality is all relative. I don't believe I ever said that morality was all relative. What I said was that I can make my m
10613,0.356,talk.religion.misc ✔,"/(hudson) /Yes you do. Who is to say that it is immoral for onesself to experience /pain or to be hurt in some other way. Maybe unpleasant, but that doesn't /say anything about morality. It violate"
590,0.330,talk.religion.misc ✔,"/(me) /(hudson) /Why does that make something immoral? Because you are not being consistent. Moral systems must be consistent. A person who thinks they can inflict pain on others, but doesn't wa"
10184,0.316,talk.religion.misc ✔,"/(hudson) /If all morality were relative- big hairy deal. As I said, you appear to be the only person saying that all morality is relative. Most people I know do hold some absolutes in their moral"
3420,0.224,rec.autos ✘,People get a life !!!!!!!!!!


Índice,Similaridad,Clase,Comienzo del texto
9736,0.453,comp.sys.mac.hardware ✔,"I have used both my serial ports with a modem and a serial printer, so I cannot use Appletalk. Is there a Ethernet to Localtalk hardware that will let me use the Ethernet port on my Q700 as a Localt"
3254,0.441,comp.graphics ✘,: |> : |> Is there any way to connect two pointing devices to one serial : |> port? I haven't tried this but I believe they would interfere : |> with each other (?) even if on
1822,0.425,comp.sys.ibm.pc.hardware ✘,Can anybody please help me with information on the use of the bi-directional printer port. I have successfully used one on a Toshiba laptop by enabling bit 0 of port 0x37f and controlling bit 7 of po
346,0.423,comp.sys.ibm.pc.hardware ✘,"] Hi there, ] ] ] : I have 1s/1p/1g I/O card in my 386/40 PC. ] : When I plug in wang modem at com4,it works. If I change ] : it to com1- it doesn't. ] : Program ""chkport"" gives diagnostics like"
5830,0.388,comp.sys.mac.hardware ✔,Battery powered devices like the PowerBook are sometimes more sensitive to serial port weirdness. I had trouble with connecting my Mac Plus to an HP 95LX handheld. Everything else worked okay on t


Índice,Similaridad,Clase,Comienzo del texto
8719,0.256,sci.med ✘,"Hello,"
2429,0.256,comp.graphics ✘,"Hello,"
342,0.246,comp.windows.x ✘,I need to have PCs and SPARCstations run the same application ( namely MicroSoft Project ). The original system ran on the PC. Now it needs to be expanded to allow UNIX users to work with the applicat
896,0.205,comp.windows.x ✘,"What ""it may think is right"" may be exactly what the user wants. Assuming that your application ""has reason to know better"" is, IMHO, anti-social. If I start your application with a -geometry optio"
3796,0.203,comp.windows.x ✘,<Dear Xperts: < < Here's my problem: I am designing several related X <applications (each one is a separate executable) and I'd like <to build a single control window that would allow the user t


Índice,Similaridad,Clase,Comienzo del texto
5084,0.111,alt.atheism ✘,": Regardless of people's hidden motivations, the stated reasons for many : wars include religion. Of course you can always claim that the REAL : reason was economics, politics, ethnic strife, or wha"
3787,0.108,alt.atheism ✘,"Well now, we can't judge death until we are dead right? So, why should we judge religion without having experienced it? People have said that religion is bad by any account, and that it is in no w"
9749,0.105,sci.med ✘,"T(> Russell Turpin responds to article by Ron Roth: T(> T(> R> ... I don't doubt that the placebo effect is alive and well with T(> R> EVERY medical modality - estimated by some to be around 20+%, T(>"
7867,0.097,talk.politics.guns ✔,"when does carrying a tool classify someone as a rambo. so all the pioneers that came west were rambo's? adrienne!!! :-) would your tune change if you were one of the ""dozen or two bear attacks""?"
10447,0.095,talk.politics.misc ✘,"Correct me if I'm wrong, but isn't this irrelevant? Either the data shows something, or it doesn't. Regardless of what other studies show. Admit it. What you SHOWED to us doesn't prove that gay"


Índice,Similaridad,Clase,Comienzo del texto
1114,0.208,talk.politics.misc ✘,"If you happen to know a political position which does not have people advocating it who do more harm than good, please point it out. One of the advantages and draw-backs of req"
3839,0.149,sci.crypt ✔,"... In reply to mail queries; I don't know if a video is available yet. I asked about a month ao and was told RSN. Several have also asked which of Chomskys books. My answer is ALL of them, and anyt"
7430,0.130,talk.politics.mideast ✘,I only want to say that I agree with Noam on this point and I hope that all sides stop targeting civilians.
3862,0.129,talk.politics.misc ✘,"Aviation Week March 15 1993 p.48 ""the CBO estimates that matching the capability of 100 C-17s would require 70 C-5s at a total cost of $14.4 billion. This option is still more than $10 billion cheap"
6751,0.126,talk.politics.misc ✘,It is also widely stated (in non-mainstream sources) that the CIA had a large part in the overthrow of the popular (and popularly-elected) left-leaning Premier Mossadegh in 1953. Is this widely reco


**Análisis:**

Se eligieron al azar los documentos 8754, 4965, 7404, 1009 y 4899, y para cada uno se buscaron los 5 documentos de train más similares con similaridad coseno sobre la representación TF-IDF. En total, **8 de los 25 vecinos (32 %) son de la misma clase** que el documento original, con similaridades bajas en general, la más alta es 0,453 y en dos documentos ningún vecino supera 0,21. Los resultados varían mucho según el documento:

| Documento | Clase | Tokens | Vecinos de la misma clase | Similaridades |
|---|---|---|---|---|
| 8754 | `talk.religion.misc` | 215 | 4/5 | 0,391 – 0,224 |
| 4965 | `comp.sys.mac.hardware` | 7 | 2/5 | 0,453 – 0,388 |
| 7404 | `comp.os.ms-windows.misc` | 14 | 0/5 | 0,256 – 0,203 |
| 1009 | `talk.politics.guns` | 29 | 1/5 | 0,111 – 0,095 |
| 4899 | `sci.crypt` | 33 | 1/5 | 0,208 – 0,126 |

**Documento 8754 (`talk.religion.misc`): buen caso.** Es el documento más largo y sus 4 primeros vecinos son de la misma clase y hablan del mismo tema, la moral y su relatividad. Además son mensajes del **mismo hilo de discusión**: todos citan a "hudson", la palabra con más peso del documento. La similaridad capta tanto el tema como la conversación. El quinto vecino, de `rec.autos`, es "People get a life!!!!!!!!!!": un documento de 4 palabras que coincide solo en "people" y "life". Al ser tan corto, esas dos palabras concentran todo su peso y alcanzan para una similaridad de 0,224.

**Documento 4965 (`comp.sys.mac.hardware`): el contenido coincide aunque la clase no.** Tiene solo 7 tokens (printer, modem, port, plug) y todos sus vecinos hablan de conectar impresoras, módems o dispositivos a puertos serie. Solo 2 son de la misma clase; los demás son de `comp.sys.ibm.pc.hardware` y `comp.graphics`. La similaridad tiene sentido por el contenido, conectar periféricos es un tema común a Mac y PC, y con tan pocas palabras no hay nada que distinga a una Mac.

**Documento 7404 (`comp.os.ms-windows.misc`): fallan los documentos cortos.** Ningún vecino es de su clase. Los dos más similares (0,256) son documentos que contienen solo "Hello,", de `sci.med` y `comp.graphics`: como el documento también es corto (14 tokens), la palabra "hello" tiene peso suficiente para que un saludo vacío sea su vecino más cercano. Los otros tres son de `comp.windows.x` y hablan de aplicaciones y ventanas: un tema cercano, porque MS Windows y X Window comparten vocabulario, pero de otra clase.

**Documento 1009 (`talk.politics.guns`): nombres propios y similaridades muy bajas.** Las palabras con más peso son nombres propios (russell, rominger, karl) y términos poco frecuentes (illogical, dogma). Como son raras, el IDF les da mucho peso, pero no describen el tema. Los vecinos coinciden en palabras como "religion" o "regardless" (dos de `alt.atheism`) o en el nombre "Russell" (un mensaje de `sci.med` que menciona a otra persona llamada Russell). Con similaridades de alrededor de 0,1, ningún documento de train es realmente parecido.

**Documento 4899 (`sci.crypt`): la etiqueta no refleja el contenido.** El mensaje habla de política y medios (Noam Chomsky, regulación de las redes), no de criptografía: está en `sci.crypt` porque forma parte de una discusión sobre privacidad en ese grupo. Sus vecinos son de clases de política, coherentes con el contenido. El único de su clase (3839) es del mismo hilo, también sobre Chomsky. Además, dos de las palabras con más peso son **errores de tipeo** ("efect", "poitical"): al aparecer en casi ningún documento, el IDF las pondera mucho, aunque no aportan nada para encontrar vecinos.

**Efecto del preprocesamiento.** Comparado con el vectorizador por defecto de sklearn, el preprocesamiento propio reduce el vocabulario de 101.631 a 55.460 términos (un 45 % menos). Para estos 5 documentos, los vecinos encontrados fueron los mismos con los dos vectorizadores, pero con similaridades más altas con el preprocesamiento propio, al eliminar stopwords, números y palabras cortas, el peso se concentra en las palabras de contenido. Como los valores de similaridad no son comparables entre representaciones distintas, su efecto sobre la clasificación se mide en el punto 2.

**Conclusiones del punto.** La similaridad coseno sobre TF-IDF tiene sentido cuando el documento es largo y usa vocabulario específico de su tema (8754). Mide **coincidencia de palabras, no de significado**, y eso trae cuatro limitaciones:

1. **Documentos cortos:** con pocas palabras, una sola coincidencia ("hello", "life") alcanza para una similaridad alta.
2. **Palabras raras sin contenido temático:** nombres propios y errores de tipeo reciben mucho peso del IDF y generan coincidencias espurias.
3. **Temas compartidos entre clases:** cuando dos grupos hablan de lo mismo (periféricos en Mac y PC, ventanas en MS Windows y X), los vecinos pueden ser correctos por contenido y de otra clase.
4. **Etiqueta vs. contenido:** la clase indica el grupo donde se publicó el mensaje, que no siempre coincide con su tema.


---
## 2. Clasificación por prototipos

In [8]:
# Funcion que devuelve un array con la clase predicha para cada documento de test.

def clasificar_por_prototipos(X_test, X_train, y_train, tam_bloque=1000):
    y_prediccion = []
    for inicio in range(0, X_test.shape[0], tam_bloque):
        similitudes = cosine_similarity(X_test[inicio:inicio + tam_bloque], X_train)   # matriz bloque × train
        y_prediccion.extend(y_train[similitudes.argmax(axis=1)])                       # clase del documento de train más similar
    return np.array(y_prediccion)

In [9]:
# Clasificamos todo test con los prototipos de train y medimos el F1-score macro
y_pred_prototipos = clasificar_por_prototipos(X_test, X_train, y_train)

f1_prototipos = f1_score(y_test, y_pred_prototipos, average="macro")
print(f"F1-score macro (clasificador por prototipos): {f1_prototipos:.3f}")

F1-score macro (clasificador por prototipos): 0.524


In [10]:
# Precisión, recall y F1 de cada una de las 20 clases
print(classification_report(y_test, y_pred_prototipos, target_names=train.target_names, digits=3))

                          precision    recall  f1-score   support

             alt.atheism      0.472     0.467     0.469       319
           comp.graphics      0.491     0.481     0.486       389
 comp.os.ms-windows.misc      0.380     0.404     0.392       394
comp.sys.ibm.pc.hardware      0.425     0.495     0.458       392
   comp.sys.mac.hardware      0.455     0.470     0.462       385
          comp.windows.x      0.596     0.509     0.549       395
            misc.forsale      0.507     0.538     0.522       390
               rec.autos      0.362     0.576     0.445       396
         rec.motorcycles      0.546     0.580     0.563       398
      rec.sport.baseball      0.667     0.688     0.677       397
        rec.sport.hockey      0.692     0.737     0.714       399
               sci.crypt      0.649     0.604     0.626       396
         sci.electronics      0.471     0.354     0.404       393
                 sci.med      0.695     0.571     0.627       396
         

In [11]:
cm = confusion_matrix(y_test, y_pred_prototipos, normalize="true")
np.fill_diagonal(cm, 0) # Se ponen en 0 los aciertos (diagonal) para quedarnos solo con las confusiones
pares = np.dstack(np.unravel_index(np.argsort(cm, axis=None)[::-1][:5], cm.shape))[0] # se eligen los 5 con mayor tasa equivocacion

# Se imprimen las 5 confusiones más frecuentes con el formato:
# clase real → clase predicha → % de los documentos de la clase real clasificados como la predicha
for real, pred in pares:
    print(f"{train.target_names[real]:28s} → {train.target_names[pred]:28s} {cm[real, pred]:.1%}")

talk.politics.misc           → talk.politics.guns           18.1%
talk.religion.misc           → soc.religion.christian       13.5%
comp.sys.mac.hardware        → comp.sys.ibm.pc.hardware     13.2%
comp.sys.ibm.pc.hardware     → comp.os.ms-windows.misc      13.0%
rec.motorcycles              → rec.autos                    12.3%


In [12]:
# Mismo clasificador con TF-IDF por defecto, para medir el efecto del preprocesamiento
X_train_default = vectorizer_default.fit_transform(train.data)
X_test_default = vectorizer_default.transform(test.data)

y_pred_default = clasificar_por_prototipos(X_test_default, X_train_default, y_train)
f1_default = f1_score(y_test, y_pred_default, average="macro")

print(f"F1 macro — preprocesamiento por defecto de sklearn: {f1_default:.4f}")
print(f"F1 macro — preprocesamiento propio:          {f1_prototipos:.4f}")

F1 macro — preprocesamiento por defecto de sklearn: 0.5050
F1 macro — preprocesamiento propio:          0.5243


**Análisis:**

El clasificador por prototipos alcanza un **F1-score macro de 0,524** y una accuracy de 0,529 con el preprocesamiento propio, frente a **0,505** con el preprocesamiento por defecto de `TfidfVectorizer`. Como el método no tiene componentes aleatorios, la mejora de casi 2 puntos se debe solo a la representación,eliminar números, stopwords y palabras cortas, y unificar formas con la lematización, hace que el documento de train más similar sea de la clase correcta con más frecuencia. El resultado es unas 10 veces mejor que el azar (≈ 0,05 con 20 clases), sin entrenar ningún modelo.

**Confusiones más frecuentes.** Las 5 confusiones con mayor proporción ocurren entre clases de temas cercanos:

- `talk.politics.misc` → `talk.politics.guns` (18,1 %)
- `talk.religion.misc` → `soc.religion.christian` (13,5 %)
- `comp.sys.mac.hardware` → `comp.sys.ibm.pc.hardware` (13,2 %)
- `comp.sys.ibm.pc.hardware` → `comp.os.ms-windows.misc` (13,0 %)
- `rec.motorcycles` → `rec.autos` (12,3 %)

Ninguna es una confusión sin sentido, en todos los casos los documentos comparten vocabulario (política, religión, hardware de computadoras, vehículos), y la frontera entre esas clases es difusa. Eso es coherente con lo observado en el punto 1, donde los vecinos de clase distinta solían ser de clases relacionadas. La confusión `rec.motorcycles` → `rec.autos` explica parte de la precision baja de `rec.autos`.

**Limitaciones.** Aun así, el clasificador se equivoca en casi la mitad de los casos. Al decidir con un único vecino, cualquier coincidencia superficial (una palabra poco frecuente compartida, un documento muy corto) lleva a una predicción errónea. Además, aunque la consigna lo describe como "tipo zero-shot" porque no hay una etapa de entrenamiento (porque no lo es), usa todos los documentos etiquetados de train como referencia, por lo que es un clasificador basado en instancias.

---
## 3. Naïve Bayes: MultinomialNB y ComplementNB

In [13]:
# Analizamos los dos modelos con parámetros por defecto sobre el TF-IDF con preprocesamiento propio
for modelo in [MultinomialNB(), ComplementNB()]:
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    print(f"{type(modelo).__name__:14s} F1 macro: {f1_score(y_test, y_pred, average='macro'):.4f}")

MultinomialNB  F1 macro: 0.6257
ComplementNB   F1 macro: 0.6779


In [14]:
# Tokenizamos una sola vez con el preprocesamiento propio, el vectorizador recibe listas de tokens ya listas
tokens_train = [preprocesar(doc) for doc in train.data]
tokens_test = [preprocesar(doc) for doc in test.data]

# Devuelve los tokens sin cambios, el preprocesamiento ya se hizo antes.
def identidad(tokens):
    return tokens

In [15]:
# Pipeline: TF-IDF + Naïve Bayes. El modelo también es un parámetro de la búsqueda.
pipeline = Pipeline([
    ("vect", TfidfVectorizer(tokenizer=identidad, preprocessor=identidad, token_pattern=None, lowercase=False)),
    ("clf", MultinomialNB()),
])

parametros = {
    "vect__min_df": [1, 2, 5],            # descartar terminos que aparecen en menos de N documentos
    "vect__max_df": [0.5, 1.0],           # descartar terminos que aparecen en más del X % de los documentos
    "vect__sublinear_tf": [False, True],  # TF lineal o 1 + log(TF)
    "clf": [MultinomialNB(), ComplementNB()],
    "clf__alpha": [0.01, 0.05, 0.1, 0.5, 1.0],   # suavizado de Laplace, para evitar palabras con probabilidad cero en una clase
}

busqueda = GridSearchCV(pipeline, parametros, scoring="f1_macro", cv=3, n_jobs=-1, verbose=1) # Se analiza el rango de parametros en ambos modelos
busqueda.fit(tokens_train, y_train) #Se entrena el modelo con los parametros ya optimos

print("Mejores parámetros:", busqueda.best_params_)
print(f"F1 macro en validación cruzada: {busqueda.best_score_:.4f}")

Fitting 3 folds for each of 120 candidates, totalling 360 fits
Mejores parámetros: {'clf': ComplementNB(), 'clf__alpha': 0.5, 'vect__max_df': 0.5, 'vect__min_df': 1, 'vect__sublinear_tf': True}
F1 macro en validación cruzada: 0.7376


In [16]:
# El mejor pipeline ya esta reentrenado con todo train, lo evaluamos una sola vez en test
mejor_modelo = busqueda.best_estimator_
y_pred_nb = mejor_modelo.predict(tokens_test)

f1_nb = f1_score(y_test, y_pred_nb, average="macro")
print(f"F1 macro en test (mejor Naïve Bayes): {f1_nb:.4f}")
print(f"F1 macro en test (prototipos):        {f1_prototipos:.4f}")
print(classification_report(y_test, y_pred_nb, target_names=train.target_names, digits=3))

F1 macro en test (mejor Naïve Bayes): 0.6834
F1 macro en test (prototipos):        0.5243
                          precision    recall  f1-score   support

             alt.atheism      0.319     0.436     0.368       319
           comp.graphics      0.708     0.666     0.686       389
 comp.os.ms-windows.misc      0.673     0.533     0.595       394
comp.sys.ibm.pc.hardware      0.614     0.668     0.640       392
   comp.sys.mac.hardware      0.747     0.706     0.726       385
          comp.windows.x      0.737     0.767     0.752       395
            misc.forsale      0.774     0.710     0.741       390
               rec.autos      0.808     0.745     0.775       396
         rec.motorcycles      0.818     0.756     0.786       398
      rec.sport.baseball      0.903     0.846     0.874       397
        rec.sport.hockey      0.856     0.940     0.896       399
               sci.crypt      0.728     0.798     0.761       396
         sci.electronics      0.684     0.557     0

In [17]:
# Mostramos los mejores resultados de las 120 combinaciones.
resultados = pd.DataFrame(busqueda.cv_results_)
resultados["modelo"] = resultados["param_clf"].apply(lambda m: type(m).__name__)
columnas = ["modelo", "param_clf__alpha", "param_vect__min_df", "param_vect__max_df",
            "param_vect__sublinear_tf", "mean_test_score"]
resultados[columnas].sort_values("mean_test_score", ascending=False).groupby("modelo").head(3)

,modelo,param_clf__alpha,param_vect__min_df,param_vect__max_df,param_vect__sublinear_tf,mean_test_score
103,ComplementNB,0.50,1,1.0,True,0.737627
97,ComplementNB,0.50,1,0.5,True,0.737627
96,ComplementNB,0.50,1,0.5,False,0.736329
0,MultinomialNB,0.01,1,0.5,False,0.731799
6,MultinomialNB,0.01,1,1.0,False,0.731799
7,MultinomialNB,0.01,1,1.0,True,0.727142


**Análisis:**

**Línea base.** Con los parámetros por defecto y el TF-IDF del preprocesamiento propio, `MultinomialNB` obtiene un F1 macro de **0,626** y `ComplementNB` de **0,678** en test. Los dos ya superan ampliamente al clasificador por prototipos (0,524), y `ComplementNB` le gana a `MultinomialNB` por 5 puntos sin ajustar nada.

**Búsqueda de parámetros.** Se buscó la mejor combinación de parámetros del vectorizador (`min_df`, `max_df`, `sublinear_tf`) y del modelo (`MultinomialNB` o `ComplementNB`, y `alpha`) con validación cruzada de 3 folds sobre train, maximizando el F1 macro (120 combinaciones, 360 entrenamientos), y el mejor pipeline se evaluó una única vez en test. Se eligió validación cruzada en lugar de comparar directamente contra test para que el F1 de test no quede sesgado por haber sido usado en la elección. Para que la búsqueda no repita el preprocesamiento en cada ajuste, los documentos se tokenizaron una sola vez con `preprocesar` y el vectorizador recibe los tokens ya listos (función `identidad`).

El mejor modelo fue **`ComplementNB` con `alpha=0,5`, `min_df=1` y `sublinear_tf=True`**, con un F1 macro de 0,738 en validación cruzada, apenas por encima del mejor `MultinomialNB` (0,732, con `alpha=0,01`). Sobre cada parámetro:

- **`max_df` no tuvo efecto** (0,5 y 1,0 dan el mismo resultado): como el preprocesamiento ya elimina las stopwords, casi ningún término aparece en más de la mitad de los documentos.
- **`min_df=1` fue el mejor valor en los dos modelos.** A diferencia del clasificador por prototipos, donde las palabras raras generaban coincidencias espurias, Naïve Bayes acumula la evidencia de todos los documentos de cada clase, y una palabra poco frecuente que aparece siempre en la misma clase es una pista útil.
- **`sublinear_tf` tuvo un efecto menor**, porque en estos mensajes cortos la mayoría de las palabras aparecen una o dos veces por documento.
- **`alpha` fue el parámetro con más influencia.** Con pesos TF-IDF (valores entre 0 y 1), el suavizado por defecto (`alpha=1`) resulta excesivo, sobre todo para `MultinomialNB`, cuyo mejor valor fue el más chico de la grilla. Eso explica la mayor parte de la diferencia entre los dos modelos en la línea base.

**Resultado en test.** El mejor modelo alcanza un **F1 macro de 0,683** y una accuracy de 0,703, frente a 0,524 y 0,529 del clasificador por prototipos: una mejora de 16 puntos. Respecto de `ComplementNB` con parámetros por defecto (0,678), la búsqueda aporta solo medio punto en test: el salto principal viene de pasar de un único vecino a un modelo que resume cada clase. 

**Desempeño por clase.** La mejora no es pareja. Las clases de vocabulario más específico superan 0,85 de F1 (`rec.sport.hockey`, `rec.sport.baseball`), pero dos clases empeoran respecto del clasificador por prototipos, `alt.atheism` (0,469 → 0,368) y `talk.religion.misc` (0,339 → 0,201), esta última con un recall de solo 0,127. `soc.religion.christian`, en cambio, tiene recall 0,887 y precision 0,559; el modelo la predice de más y absorbe documentos de las otras clases de religión. Lo mismo ocurre en política, donde `talk.politics.guns` (precision 0,585) absorbe parte de `talk.politics.misc` (recall 0,423). Naïve Bayes aprende un perfil de vocabulario por clase, y cuando dos clases comparten casi todo su vocabulario, la de tema más específico y consistente se impone sobre la más general (`misc`), que queda sin palabras propias que la identifiquen.

---
## 4. Similaridad entre palabras (matriz término-documento)

In [18]:
# Matriz término-documento: cada fila es una palabra, cada columna un documento
X_terminos = X_train.T.tocsr()
print(f"Matriz término-documento: {X_terminos.shape[0]} términos × {X_terminos.shape[1]} documentos")

# Palabras elegidas a mano, de temas distintos (en su forma lematizada, como quedaron en el vocabulario)
palabras = ["car", "god", "space", "windows", "gun"]

for palabra in palabras:
    if palabra not in vectorizer.vocabulary_:
        print(f"'{palabra}' no está en el vocabulario")
        continue
    idx = vectorizer.vocabulary_[palabra]
    idx_similares, sims = top_k_similares(X_terminos, idx, k=5)
    similares = ", ".join(f"{idx2word[j]} ({s:.3f})" for j, s in zip(idx_similares, sims))
    print(f"{palabra:10s} → {similares}")

Matriz término-documento: 55460 términos × 11314 documentos
car        → civic (0.194), dealer (0.164), mile (0.162), owner (0.144), volvo (0.143)
god        → bible (0.279), jesus (0.279), faith (0.269), christ (0.260), existence (0.258)
space      → nasa (0.310), shuttle (0.284), exploration (0.241), aa (0.238), seti (0.233)
'windows' no está en el vocabulario
gun        → crime (0.264), firearm (0.256), criminal (0.255), handgun (0.232), homicide (0.218)


**Análisis:**

Al transponer la matriz documento-término se obtiene una matriz término-documento de 55.460 términos × 11.314 documentos, donde cada palabra queda representada por un vector con su peso TF-IDF en cada documento. La similaridad coseno entre dos de esos vectores mide en qué medida las dos palabras aparecen en los mismos documentos.

Se eligieron a mano 5 palabras de temas distintos: "car", "god", "space", "window" y "gun". Para todas, los 5 términos más similares son claramente del mismo tema:

| Palabra | Términos más similares | Tema |
|---|---|---|
| car | civic, dealer, mile, owner, volvo | Modelos de auto, concesionaria, kilometraje |
| god | bible, jesus, faith, christ, existence | Religión cristiana y existencia de Dios (`soc.religion.christian`, `alt.atheism`) |
| space | nasa, shuttle, exploration, aa, seti | Agencia espacial, transbordador, búsqueda de vida extraterrestre |
| window | manager, do, file, application, running | Sistemas operativos e interfaces |
| gun | crime, firearm, criminal, handgun, homicide | Armas y criminalidad (`talk.politics.guns`) |

**Similaridad temática, no semántica.** La similaridad refleja **coaparición en los mismos documentos**, no equivalencia de significado. "Civic" y "volvo" no son sinónimos de "car", sino palabras que aparecen en los mismos mensajes. Incluso cuando aparecen sinónimos, como "firearm" y "handgun" para "gun", se debe a que se usan en las mismas discusiones y no a que el método capte su significado. Los valores de similaridad son bajos (entre 0,14 y 0,31), porque cada palabra aparece en muchos documentos y basta con compartir una parte de ellos para quedar entre las más similares.

**Efectos del preprocesamiento.** "windows" no figura en el vocabulario porque el lematizador de WordNet la convirtió en "window", unificando el sistema operativo con la ventana. Aun así, todos sus términos similares son de computación, en este corpus la palabra se usa casi siempre en referencia a Windows o a X Window, y "manager" proviene de "Program Manager" y "window manager".


---
## Conclusiones

| Método | F1 macro en test |
|---|---|
| Clasificador por prototipos, preprocesamiento por defecto de sklearn | 0,505 |
| Clasificador por prototipos, preprocesamiento propio | 0,524 |
| `MultinomialNB` con parámetros por defecto | 0,626 |
| `ComplementNB` con parámetros por defecto | 0,678 |
| **`ComplementNB` optimizado** (`alpha=0,5`, `min_df=1`, `sublinear_tf=True`) | **0,683** |

1. **La representación TF-IDF captura bien el tema de los documentos, pero no su significado.** La similaridad coseno funciona con documentos largos y de vocabulario específico, y falla con documentos cortos, nombres propios, errores de tipeo y temas compartidos entre clases. Lo mismo se observa entre palabras: los términos más similares son del mismo tema por coaparición, no sinónimos.

2. **El preprocesamiento ayuda, aunque su efecto es moderado.** El pipeline de la clase 1 (minúsculas, tokenización con NLTK, eliminación de puntuación, números, stopwords y palabras cortas, y lematización) reduce el vocabulario un 45 % y mejora el F1 del clasificador por prototipos en 2 puntos. También tiene costos: la lematización sin categoría gramatical unifica "windows" con "window" y deforma siglas como DOS y AAS.

3. **Resumir cada clase es mucho mejor que comparar con un único vecino.** Naïve Bayes supera al clasificador por prototipos en 16 puntos de F1 macro, porque acumula la evidencia de todos los documentos de cada clase y no depende de una coincidencia puntual. La mayor parte de la mejora la da el cambio de método: la búsqueda de parámetros aporta poco sobre `ComplementNB` por defecto, y el parámetro más importante es el suavizado `alpha`.

4. **Los errores se concentran en clases de temas cercanos**, con los dos métodos: religión (`talk.religion.misc`, `alt.atheism`, `soc.religion.christian`), política (`talk.politics.misc` y `talk.politics.guns`) y computación (las clases `comp.*`). En particular, las clases generales (`misc`) son las más difíciles, porque no tienen vocabulario propio, y con Naïve Bayes quedan absorbidas por las clases más específicas que comparten su tema.
